# M5 Fed Speech Hawkish/Dovish Classifier — Demo

Pipeline: scrape (or synthetic) → lexicon-label → TF-IDF + LogisticRegression → backtest vs 2y yield → charts.

Run order: cells top-to-bottom. Pass `SYNTHETIC=True` to skip the network scrape.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)
SYNTHETIC = True   # flip to False to attempt live federalreserve.gov scrape
print('repo root:', ROOT)

In [ ]:
from m5_fed_speech import scrape
out = Path('data/speeches.csv')
if SYNTHETIC:
    scrape.write_synthetic(out)
else:
    scrape.scrape_years(range(2015, 2025), out)
import pandas as pd
df = pd.read_csv(out)
df.head()

In [ ]:
from m5_fed_speech.classify import train, label_corpus
labeled = label_corpus(df)
labeled[['date','title','score','label']].head()

In [ ]:
res = train(df)
print(f'accuracy={res.accuracy:.4f}  n_train={res.n_train}  n_test={res.n_test}')
labeled.to_csv('data/speeches_labeled.csv', index=False)

In [ ]:
from m5_fed_speech import backtest
yields, source = backtest.load_yields()
merged = backtest.compute_yield_changes(labeled, yields, horizon_days=30)
metrics = backtest.metrics(merged)
print('yield source:', source)
print(metrics)
merged.to_csv('data/backtest.csv', index=False)

In [ ]:
from m5_fed_speech import charts
from pathlib import Path
charts.histogram_hawk_dove(labeled, Path('docs/hist_hawk_dove.png'))
charts.scatter_score_vs_yield(merged, Path('docs/scatter_score_yield.png'))
from IPython.display import Image, display
display(Image('docs/hist_hawk_dove.png'))
display(Image('docs/scatter_score_yield.png'))